# *Being and Time* → Markdown, on a Colab GPUTranscribes the scanned PDF using [marker](https://github.com/datalab-to/marker).**Why this notebook exists.** On an Apple M2 the run takes ~7 hours, because thesurya vision model generates ~1,500 tokens per page and every token re-reads thewhole 1.3 GB model from memory — it is memory-bandwidth bound. A T4 has roughly3x the bandwidth *and* enough spare VRAM to OCR several pages at once, which theMac never had.**Two Colab-specific obstacles, both handled below:**1. marker's default NVIDIA backend spawns a **Docker** image. Colab has no Docker   daemon. We force the `llamacpp` backend instead, which offloads all layers to   CUDA via `-ngl 99` — the same code path already validated on the Mac.2. llama.cpp ships prebuilt CUDA binaries **for Windows only**. So we compile   `llama-server` here (~5–10 min) and **cache it to Drive**, so later sessions   skip the build.Run the cells in order.

## 1. Confirm a GPU runtimeIf this fails: **Runtime → Change runtime type → T4 GPU**, then rerun.

In [ ]:
!nvidia-smi || echo "NO GPU -- set Runtime > Change runtime type > T4 GPU"

## 2. Mount DriveDrive holds three things, all of which should survive a session reset:the source PDF, the compiled `llama-server`, and the output (so an interruptedrun resumes instead of restarting).

In [ ]:
from google.colab import drivedrive.mount('/content/drive')

## 3. Configure paths**Before running this**, copy the project folder to Drive — everything except`.venv/` and `output/` (it is only a few hundred KB), plus the source PDF.Expected layout in Drive:```MyDrive/being-and-time/    bt/                 <- the pipeline package    pyproject.toml    42700894-Martin-Heidegger-Being-and-Time.pdf```Outputs and the cached binary are written alongside it.

In [ ]:
PROJECT = '/content/drive/MyDrive/being-and-time'   # project folder in DrivePDF     = f'{PROJECT}/42700894-Martin-Heidegger-Being-and-Time.pdf'OUTDIR  = f'{PROJECT}/output'                       # resumable: keep on DriveCACHE   = f'{PROJECT}/llama-cache'                  # compiled llama-serverimport os, sysassert os.path.isdir(PROJECT), f'not found: {PROJECT}'assert os.path.isfile(PDF), f'PDF not found: {PDF}'os.makedirs(OUTDIR, exist_ok=True)# Import the pipeline from Drive. Working dir stays on local disk: marker writes# scratch files, and Drive I/O is slow.sys.path.insert(0, PROJECT)os.makedirs('/content/work', exist_ok=True)os.chdir('/content/work')print('project :', PROJECT)print('output  :', OUTDIR)

## 4. Install dependenciesColab already ships a CUDA build of torch. marker needs `torch>=2.7,<3`; if thepreinstalled one satisfies that, pip leaves it alone — do **not** force areinstall, or you may end up replacing a working CUDA torch.

In [ ]:
!pip install -q marker-pdf pymupdfimport torchprint('torch', torch.__version__, '| cuda', torch.cuda.is_available(), '|', torch.cuda.get_device_name(0))

## 5. Build the CUDA `llama-server`First run compiles it (~5–10 min) and caches it to Drive. Later sessions reusethe cached binary in seconds.

In [ ]:
from bt import gpu_setupgpu_setup.report()env = gpu_setup.configure(cache_dir=CACHE)   # builds if not cached, sets env vars

## 6. Download the models (~1.8 GB)Fetched up front deliberately. marker starts each model in its own serversubprocess with a 300s health check — on a cold cache the download outlives thecheck and marker force-kills its own server with a confusing `SpawnError`.`HF_HUB_DISABLE_XET=1` is already set by the previous cell; without it Hubdownloads can hang at 0 bytes indefinitely on some networks.

In [ ]:
from bt.warmup import warm_allwarm_all()

## 7. Split the 2-page spreadsEach PDF page is a scanned spread of two facing book pages. They are cut apartfirst, at a gutter detected per page (it drifts between 0.497 and 0.514, so afixed 50% split would clip text).Expect **582** pages out of 294 spreads — six halves are blank.

In [ ]:
from pathlib import Pathfrom bt.split_spreads import split_documentsplit_pdf = Path(OUTDIR) / 'pages.pdf'records = split_document(Path(PDF), split_pdf, Path(OUTDIR) / 'pagemap.json')print(f'{len(records)} book pages -> {split_pdf}')

## 8. Quick sanity check (optional but recommended)Two pages through the full pipeline before committing to the whole book. Page 0is the opening page: dense polytonic Greek plus a half-page footnote block — thehardest content in the document.The key check is **fresh OCR**: the PDF carries a bad Acrobat OCR layer, and ifmarker ever reads that instead of OCRing, you get readable Markdown made of thewrong characters. `verify` greps for that layer's distinctive damage.

In [ ]:
import timefrom bt.transcribe import transcribefrom bt.verify import run_allt0 = time.time()transcribe(split_pdf, Path('/content/work/sample.md'),           Path('/content/work/chunks-sample'), total_pages=2, chunk_size=2)print(f'--- {(time.time()-t0)/2:.1f}s per page ---')for f in run_all(Path('/content/work/sample.md').read_text()):    print(f"[{'ok  ' if f.ok else 'FAIL'}] {f.name:<13} {f.detail}")

## 9. Transcribe the bookChunks are written to Drive as they finish, so this is **resumable**: if Colabdisconnects, just rerun this cell and it skips completed chunks.Free-tier sessions idle out after ~90 minutes, so keep the tab active.

In [ ]:
raw_md = Path(OUTDIR) / 'raw.md'transcribe(    split_pdf, raw_md,    Path(OUTDIR) / 'chunks',    total_pages=len(records),    chunk_size=10,        # smaller chunks = finer-grained resume    dpi=300,              # matches the scan; lowering it does NOT speed things up)

## 10. Clean up and verifyStrips running heads, namespaces footnote ids per page (footnote "1" recurs onnearly every page, so un-namespaced ids would collide hundreds of times), andrejoins words hyphenated across line breaks.

In [ ]:
from bt.postprocess import processfinal_md = Path(OUTDIR) / 'being-and-time.md'cleaned, stats = process(raw_md.read_text(encoding='utf-8'))final_md.write_text(cleaned, encoding='utf-8')print(stats.render())print()for f in run_all(cleaned):    print(f"[{'ok  ' if f.ok else 'FAIL'}] {f.name:<13} {f.detail}")print(f'\nWrote {final_md} ({len(cleaned):,} chars)')

## 11. DownloadThe file is already on Drive; this is just a direct download.

In [ ]:
from google.colab import filesfiles.download(str(final_md))

---### Troubleshooting| Symptom | Cause / fix ||---|---|| `No CUDA GPU visible` | Runtime → Change runtime type → T4 GPU || Build fails, `nvcc not found` | Not on a GPU runtime || `SpawnError: ... failed to become healthy` | Models weren't cached — rerun cell 6 || Download stuck at 0 bytes | `HF_HUB_DISABLE_XET=1` missing; rerun cell 5 || Session disconnected | Rerun cell 9 — completed chunks are skipped || Out of VRAM | `gpu_setup.configure(cache_dir=CACHE, parallel=2)` |Lowering `dpi` is **not** a speed fix: 192 vs 300 DPI measured 92.1 vs82.5–95.2 s/page on the Mac for 99.71% identical output. Cost is dominated bytokens generated, not pixels read.